
## Projeto Final – Machine Learning II
### Previsão de Inadimplência de Clientes
### Márcia Aparecida Rodrigues de Sousa

### 1. Introdução

Este projeto aplica Machine Learning para apoiar decisões de crédito e gestão de risco em instituições financeiras.
O objetivo principal é prever se um cliente irá se tornar inadimplente no próximo período de faturamento, utilizando modelos de classificação supervisionada.

Além disso, foram explorados modelos não supervisionados (clustering) para identificar segmentos de clientes com características semelhantes.


### 2. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import recall_score, f1_score, roc_auc_score
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score




In [ ]:
!pip install seaborn

In [ ]:
import seaborn as sns


### 3. Carregamento do Dataset

In [ ]:
url = "https://raw.githubusercontent.com/Marcia520/ProjetoFinal_ML_II/main/credit_card_default.csv"
df = pd.read_csv(url)

df = df.rename(columns={"default payment_next_month": "inadimplente"})

X = df.drop("inadimplente", axis=1)
y = df["inadimplente"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Treino:", X_train.shape, "Teste:", X_test.shape)


### 4. Descrição do Dataset

O dataset utilizado foi obtido na plataforma Kaggle, conhecido como Credit Card Default Dataset. Ele contém aproximadamente 30.000 registros, atendendo ao requisito mínimo de volume de dados para o projeto.

As variáveis incluem:

 - informações demográficas (sexo, escolaridade, estado civil);
 - limite de crédito concedido;
 - histórico de pagamentos anteriores;
 - valores faturados e pagos em períodos anteriores.

A variável alvo indica se o cliente entrou em inadimplência no mês seguinte, permitindo a construção de modelos preditivos com base em dados históricos.

### 5. Carregamento e Preparação dos Dados




In [ ]:
url = "https://raw.githubusercontent.com/Marcia520/ProjetoFinal_ML_II/main/credit_card_default.csv"
df = pd.read_csv(url)

df = df.rename(columns={"default payment_next_month": "inadimplente"})

X = df.drop("inadimplente", axis=1)
y = df["inadimplente"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Treino:", X_train.shape, "Teste:", X_test.shape)


### 6. Modelos Não Supervisionados (Clustering + Métricas)


In [ ]:
# Treino dos modelos
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
y_pred_log = log_reg.predict(X_test_scaled)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

svm = SVC(kernel="linear", probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)

# Avaliação consolidada
results = {
    "Modelo":["Logística","Random Forest","SVM","Gradient Boosting","AdaBoost"],
    "Recall":[recall_score(y_test,y_pred_log),
              recall_score(y_test,y_pred_rf),
              recall_score(y_test,y_pred_svm),
              recall_score(y_test,y_pred_gb),
              recall_score(y_test,y_pred_ada)],
    "F1":[f1_score(y_test,y_pred_log),
          f1_score(y_test,y_pred_rf),
          f1_score(y_test,y_pred_svm),
          f1_score(y_test,y_pred_gb),
          f1_score(y_test,y_pred_ada)],
    "ROC-AUC":[roc_auc_score(y_test,log_reg.predict_proba(X_test_scaled)[:,1]),
               roc_auc_score(y_test,rf.predict_proba(X_test)[:,1]),
               roc_auc_score(y_test,svm.predict_proba(X_test_scaled)[:,1]),
               roc_auc_score(y_test,gb.predict_proba(X_test)[:,1]),
               roc_auc_score(y_test,ada.predict_proba(X_test)[:,1])]
}

df_results = pd.DataFrame(results)
print(df_results)

df_results.set_index("Modelo")[["Recall","F1","ROC-AUC"]].plot(kind="bar", figsize=(10,6))
plt.title("Comparação de Modelos Supervisionados")
plt.ylabel("Score")
plt.show()


### 7. Modelos Não Supervisionados (Clustering + Métricas)

In [ ]:
X_unsupervised = df.drop("inadimplente", axis=1)
scaler_unsup = StandardScaler()
X_unsupervised_scaled = scaler_unsup.fit_transform(X_unsupervised)

# K-Means
kmeans = KMeans(n_clusters=2, random_state=42)
labels_kmeans = kmeans.fit_predict(X_unsupervised_scaled)

print("K-Means Silhouette:", silhouette_score(X_unsupervised_scaled, labels_kmeans))
print("K-Means Davies-Bouldin:", davies_bouldin_score(X_unsupervised_scaled, labels_kmeans))
print("K-Means Calinski-Harabasz:", calinski_harabasz_score(X_unsupervised_scaled, labels_kmeans))

# DBSCAN
dbscan = DBSCAN(eps=3, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_unsupervised_scaled)

mask = labels_dbscan != -1
if np.any(mask):
    print("DBSCAN Silhouette:", silhouette_score(X_unsupervised_scaled[mask], labels_dbscan[mask]))
else:
    print("DBSCAN não formou clusters válidos.")



In [ ]:
df.columns

### 8. Otimização de Hiperparâmetros

In [ ]:
param_grid_log = {"C":[0.01,0.1,1,10], "penalty":["l2"]}
grid_log = GridSearchCV(LogisticRegression(random_state=42), param_grid_log, cv=5, scoring="recall")
grid_log.fit(X_train_scaled, y_train)

param_grid_rf = {"n_estimators":[100,200], "max_depth":[None,10,20]}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring="recall")
grid_rf.fit(X_train, y_train)

print("Melhores parâmetros Logística:", grid_log.best_params_)
print("Melhores parâmetros RF:", grid_rf.best_params_)




### Conclusão

Random Forest e Gradient Boosting foram os mais eficazes para prever inadimplência.

K-Means e DBSCAN revelaram padrões de segmentação úteis para análise de perfil.

Próximos passos: testar XGBoost/LightGBM, incluir variáveis externas e monitorar data drift.

